##### Outpatient Clinic Patient-Flow and Staffing Optimisation

##### This case study uses synthetic outpatient-clinic data to evaluate doctor staffing across four time blocks. The analysis compares patient consultation workload with available doctor capacity, identifies periods of workload pressure, evaluates alternative staffing scenarios and recommends a resource-allocation approach that balances waiting-time risk, utilisation and staffing cost.

In [46]:
import pandas as pd
import numpy as np
from datetime import datetime

#### Load data

In [47]:
df_dr = pd.read_csv('doctor_availability_synthetic.csv')
df_patient = pd.read_csv('patient_visits_synthetic.csv')
df_staff = pd.read_csv('staffing_scenarios_synthetic.csv')
df_model = pd.read_csv('model_parameters_synthetic.csv')

##### Doctor Availability Synthetic Data

In [48]:
df_dr

,clinic_day,weekday,time_block,min_doctors_required,max_doctors_available,doctor_cost_per_2hr_block,target_p90_wait_minutes
0,2026-05-04,Monday,08:00-10:00,1,3,240,45
1,2026-05-04,Monday,10:00-12:00,1,3,240,45
2,2026-05-04,Monday,13:00-15:00,1,3,240,45
3,2026-05-04,Monday,15:00-17:00,1,3,240,45
4,2026-05-05,Tuesday,08:00-10:00,1,3,240,45
...,...,...,...,...,...,...,...
95,2026-06-04,Thursday,15:00-17:00,1,3,240,45
96,2026-06-05,Friday,08:00-10:00,1,3,240,45
97,2026-06-05,Friday,10:00-12:00,1,3,240,45
98,2026-06-05,Friday,13:00-15:00,1,3,240,45


##### Data Quality Check for Doctor Capacity by Day and Time Block

In [49]:
df_dr.isna().sum().sort_values(ascending=False)

clinic_day                   0
weekday                      0
time_block                   0
min_doctors_required         0
max_doctors_available        0
doctor_cost_per_2hr_block    0
target_p90_wait_minutes      0
dtype: int64

##### Data Quality Checks for Patient Demand, Arrival Patterns, Consultation Duration, No-Show Behaviour and Patient Mix

In [50]:
df_all_patients = df_patient.copy()
df_patient.isna().sum()

patient_id                 0
clinic_day                 0
weekday                    0
clinic_site                0
appointment_time           0
appointment_minute         0
time_block                 0
no_show                    0
actual_arrival_time      134
actual_arrival_minute    134
arrival_delay_minutes    134
priority                   0
age_band                   0
visit_type                 0
service_type               0
complexity                 0
skill_required             0
registration_minutes     134
consultation_minutes     134
dtype: int64

#### Staffing Scenario Synthetic Data

In [51]:
df_staff

,scenario,08:00-10:00,10:00-12:00,13:00-15:00,15:00-17:00,description
0,Baseline_2_Doctors_All_Day,2,2,2,2,Current-state baseline with two doctors throug...
1,Extra_3_Doctors_All_Day,3,3,3,3,Adds one doctor across all clinic blocks; stro...
2,Peak_3_Doctors_Morning,3,3,2,2,Adds capacity only in morning peak blocks.
3,Lean_Low_Demand_Afternoon,2,2,2,1,Reduces capacity in late afternoon to save cos...
4,Balanced_Peak_Only,3,2,2,2,Adds one doctor only during the busiest early ...


##### Patient Visits Synthetic Data

In [52]:
df_patient

,patient_id,clinic_day,weekday,clinic_site,appointment_time,appointment_minute,time_block,no_show,actual_arrival_time,actual_arrival_minute,arrival_delay_minutes,priority,age_band,visit_type,service_type,complexity,skill_required,registration_minutes,consultation_minutes
0,P00001,2026-05-04,Monday,Clinic A,08:00,480,08:00-10:00,0,07:45,465.0,-15.0,Routine,Adult,New,General Review,Low,General,6.0,13.4
1,P00002,2026-05-04,Monday,Clinic A,08:08,488,08:00-10:00,0,07:59,479.0,-9.0,Routine,Adult,Follow-up,Complex Care Review,Medium,Complex,5.1,48.0
2,P00003,2026-05-04,Monday,Clinic A,08:16,496,08:00-10:00,0,08:15,495.0,-1.0,Routine,Adult,Follow-up,Physiotherapy Review,Low,Rehab,7.8,30.8
3,P00004,2026-05-04,Monday,Clinic A,08:24,504,08:00-10:00,0,08:17,497.0,-7.0,Routine,Adult,Follow-up,Complex Care Review,Low,Complex,5.8,40.3
4,P00005,2026-05-04,Monday,Clinic B,08:32,512,08:00-10:00,0,08:36,516.0,4.0,Routine,Adult,Follow-up,Post-op Follow-up,Low,General,5.3,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1345,P01346,2026-06-05,Friday,Clinic A,16:10,970,15:00-17:00,1,NaN,NaN,NaN,Routine,Adult,New,Post-op Follow-up,Medium,General,NaN,NaN
1346,P01347,2026-06-05,Friday,Clinic A,16:20,980,15:00-17:00,0,16:23,983.0,3.0,Urgent,Adult,Follow-up,Post-op Follow-up,Medium,General,5.5,29.4
1347,P01348,2026-06-05,Friday,Clinic A,16:30,990,15:00-17:00,0,16:31,991.0,1.0,Routine,Senior,New,Physiotherapy Review,Medium,Rehab,5.5,31.4
1348,P01349,2026-06-05,Friday,Clinic B,16:40,1000,15:00-17:00,0,16:44,1004.0,4.0,Routine,Adult,New,Post-op Follow-up,Medium,General,5.2,17.8


##### Patients recorded as no-show do not have actual arrival, registration or consultation values. These missing values are expected and reflect that the patients did not attend their requirements.

In [53]:
df_patient.isna().sum().sort_values(ascending=False)
df_col = ['actual_arrival_minute', 'actual_arrival_time', 'registration_minutes', 'arrival_delay_minutes', 'consultation_minutes']
df_patient.loc[df_patient[df_col].isna().all(axis=1)]

,patient_id,clinic_day,weekday,clinic_site,appointment_time,appointment_minute,time_block,no_show,actual_arrival_time,actual_arrival_minute,arrival_delay_minutes,priority,age_band,visit_type,service_type,complexity,skill_required,registration_minutes,consultation_minutes
10,P00011,2026-05-04,Monday,Clinic A,09:20,560,08:00-10:00,1,NaN,NaN,NaN,Routine,Senior,New,Post-op Follow-up,High,General,NaN,NaN
16,P00017,2026-05-04,Monday,Clinic A,10:08,608,10:00-12:00,1,NaN,NaN,NaN,Routine,Adult,Follow-up,Post-op Follow-up,High,General,NaN,NaN
30,P00031,2026-05-04,Monday,Clinic A,13:00,780,13:00-15:00,1,NaN,NaN,NaN,Routine,Senior,Follow-up,General Review,Medium,General,NaN,NaN
40,P00041,2026-05-04,Monday,Clinic B,14:40,880,13:00-15:00,1,NaN,NaN,NaN,Routine,Senior,Follow-up,General Review,Low,General,NaN,NaN
43,P00044,2026-05-04,Monday,Clinic B,15:10,910,15:00-17:00,1,NaN,NaN,NaN,Urgent,Adult,Follow-up,Physiotherapy Review,Low,Rehab,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1333,P01334,2026-06-05,Friday,Clinic B,14:10,850,13:00-15:00,1,NaN,NaN,NaN,Routine,Senior,New,Post-op Follow-up,Low,General,NaN,NaN
1335,P01336,2026-06-05,Friday,Clinic A,14:30,870,13:00-15:00,1,NaN,NaN,NaN,Routine,Adult,New,General Review,Low,General,NaN,NaN
1337,P01338,2026-06-05,Friday,Clinic B,14:50,890,13:00-15:00,1,NaN,NaN,NaN,Urgent,Adult,Follow-up,Physiotherapy Review,Low,Rehab,NaN,NaN
1342,P01343,2026-06-05,Friday,Clinic A,15:40,940,15:00-17:00,1,NaN,NaN,NaN,Routine,Adult,Follow-up,Post-op Follow-up,Medium,General,NaN,NaN


##### No-show records were excluded from consultation workload and queue simulation because these patients did not enter the service process. Patient IDs were also checked for duplicates.

In [54]:
df_patient = df_patient.dropna(subset=['actual_arrival_minute'])
df_patient['patient_id'].duplicated().sum()

0

##### Appointment Demand Including No-shows

In [55]:
df_model

,parameter,value,unit,notes
0,doctor_cost_per_hour,120,SGD,Synthetic cost assumption for demonstration.
1,waiting_time_penalty_per_patient_minute,2,SGD-equivalent,Synthetic service penalty used in objective fu...
2,overtime_penalty_per_minute,10,SGD-equivalent,Synthetic penalty for finishing after clinic h...
3,clinic_start_minute,480,minute of day,08:00.
4,clinic_end_minute,1020,minute of day,17:00.
5,registration_counter_count,1,counter,Registration capacity in simulation.
6,service_level_target_p90_wait,45,minutes,Illustrative service-level target.


#### Operational Analysis

- Where is the demand highest?
- Which time block has the most attended patients?
- Which service type takes the longest?
- What is the no-show rate?
- Which time blocks are likely to create workload pressure?


#### Clinic A records the highest appointment demand, particularly during the 08:00-10:00 and 10:00-12:00 time blocks

In [56]:
df_all_patients['patient_id'].duplicated().sum()
highest_demand = df_all_patients.groupby(['clinic_day', 'weekday', 'time_block'])['appointment_minute'].count()
highest_demand.sort_values(ascending=False)

clinic_day  weekday    time_block 
2026-05-04  Monday     08:00-10:00    15
2026-05-13  Wednesday  10:00-12:00    15
2026-05-14  Thursday   08:00-10:00    15
2026-06-01  Monday     10:00-12:00    15
2026-05-15  Friday     08:00-10:00    15
                                      ..
2026-05-21  Thursday   15:00-17:00    12
2026-05-22  Friday     13:00-15:00    12
                       15:00-17:00    12
2026-05-25  Monday     13:00-15:00    12
2026-06-05  Friday     15:00-17:00    12
Name: appointment_minute, Length: 100, dtype: int64

In [57]:
high_demand = df_patient[['clinic_day', 'weekday', 'clinic_site', 'time_block', 'appointment_minute']]
high_demand_appt = high_demand.groupby(['clinic_day', 'weekday', 'clinic_site', 'time_block'])['appointment_minute'].count()
high_demand_appt.sort_values(ascending=False).head(20)

clinic_day  weekday    clinic_site  time_block 
2026-05-26  Tuesday    Clinic A     08:00-10:00    13
2026-05-21  Thursday   Clinic A     08:00-10:00    12
                                    10:00-12:00    12
2026-05-12  Tuesday    Clinic A     10:00-12:00    11
2026-05-06  Wednesday  Clinic A     10:00-12:00    11
2026-05-19  Tuesday    Clinic A     08:00-10:00    11
2026-05-15  Friday     Clinic B     10:00-12:00    11
2026-05-25  Monday     Clinic A     08:00-10:00    11
2026-05-13  Wednesday  Clinic A     10:00-12:00    11
2026-05-08  Friday     Clinic A     10:00-12:00    11
2026-06-02  Tuesday    Clinic A     08:00-10:00    11
2026-05-07  Thursday   Clinic A     08:00-10:00    11
2026-05-20  Wednesday  Clinic A     13:00-15:00    10
2026-05-12  Tuesday    Clinic A     08:00-10:00    10
2026-06-01  Monday     Clinic A     10:00-12:00    10
2026-05-07  Thursday   Clinic A     10:00-12:00    10
2026-05-04  Monday     Clinic A     08:00-10:00    10
2026-05-05  Tuesday    Clinic A   

#### Complex Care Review has the highest average consultation minutes

In [58]:
df_patient.groupby('service_type')['consultation_minutes'].mean().sort_values(ascending=False).round(2)

service_type
Complex Care Review     38.33
Physiotherapy Review    28.75
Post-op Follow-up       21.36
General Review          15.31
Name: consultation_minutes, dtype: float64

##### Around 10% of no show rate for patients

In [59]:
no_show_count = len(df_all_patients[df_all_patients['no_show'] == 1])
total_patients = len(df_all_patients)
no_show_rate = 100.0 * no_show_count / total_patients
round(no_show_rate, 2)

9.93

#### Convert the time block into minutes

In [60]:
time_block_start = datetime.strptime("08:00", "%H:%M")
time_block_end = datetime.strptime("10:00", "%H:%M")

df_all_patients['hours_spent'] = time_block_end - time_block_start
df_all_patients['time_block_mins_spent'] = df_all_patients['hours_spent'].dt.total_seconds() / 60
df_all_patients

,patient_id,clinic_day,weekday,clinic_site,appointment_time,appointment_minute,time_block,no_show,actual_arrival_time,actual_arrival_minute,...,priority,age_band,visit_type,service_type,complexity,skill_required,registration_minutes,consultation_minutes,hours_spent,time_block_mins_spent
0,P00001,2026-05-04,Monday,Clinic A,08:00,480,08:00-10:00,0,07:45,465.0,...,Routine,Adult,New,General Review,Low,General,6.0,13.4,0 days 02:00:00,120.0
1,P00002,2026-05-04,Monday,Clinic A,08:08,488,08:00-10:00,0,07:59,479.0,...,Routine,Adult,Follow-up,Complex Care Review,Medium,Complex,5.1,48.0,0 days 02:00:00,120.0
2,P00003,2026-05-04,Monday,Clinic A,08:16,496,08:00-10:00,0,08:15,495.0,...,Routine,Adult,Follow-up,Physiotherapy Review,Low,Rehab,7.8,30.8,0 days 02:00:00,120.0
3,P00004,2026-05-04,Monday,Clinic A,08:24,504,08:00-10:00,0,08:17,497.0,...,Routine,Adult,Follow-up,Complex Care Review,Low,Complex,5.8,40.3,0 days 02:00:00,120.0
4,P00005,2026-05-04,Monday,Clinic B,08:32,512,08:00-10:00,0,08:36,516.0,...,Routine,Adult,Follow-up,Post-op Follow-up,Low,General,5.3,16.0,0 days 02:00:00,120.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1345,P01346,2026-06-05,Friday,Clinic A,16:10,970,15:00-17:00,1,NaN,NaN,...,Routine,Adult,New,Post-op Follow-up,Medium,General,NaN,NaN,0 days 02:00:00,120.0
1346,P01347,2026-06-05,Friday,Clinic A,16:20,980,15:00-17:00,0,16:23,983.0,...,Urgent,Adult,Follow-up,Post-op Follow-up,Medium,General,5.5,29.4,0 days 02:00:00,120.0
1347,P01348,2026-06-05,Friday,Clinic A,16:30,990,15:00-17:00,0,16:31,991.0,...,Routine,Senior,New,Physiotherapy Review,Medium,Rehab,5.5,31.4,0 days 02:00:00,120.0
1348,P01349,2026-06-05,Friday,Clinic B,16:40,1000,15:00-17:00,0,16:44,1004.0,...,Routine,Adult,New,Post-op Follow-up,Medium,General,5.2,17.8,0 days 02:00:00,120.0


##### Workload gap is calculated as total consultation demand minus available doctor capcity. Positive values indicate capacity shortage, while negative values indicate spare capacity. The results suggest that some Tuesday afternoon blocks have lower utilisation and may provide opportunities for doctor reallocation, subject to operational and clinical constraints.

In [61]:
wp = df_all_patients.merge(df_dr, left_on=['clinic_day', 'weekday', 'time_block'], right_on=['clinic_day', 'weekday', 'time_block'], how='left')
total_consultation = wp[['clinic_day', 'weekday', 'clinic_site', 'time_block', 'consultation_minutes', 'max_doctors_available', 'time_block_mins_spent']].copy()
total_consultation['doctor_minutes'] = total_consultation['max_doctors_available'] * total_consultation['time_block_mins_spent']
total_consultation['utilization_rate'] = total_consultation['consultation_minutes'] / total_consultation['doctor_minutes']
summary = total_consultation.groupby(['clinic_day', 'weekday', 'time_block']).agg(total_consultation_minutes=('consultation_minutes', 'sum'), max_doctors=('max_doctors_available', 'max'), doctor_minutes=('doctor_minutes', 'max')).reset_index()
summary['workload_gap'] = summary['total_consultation_minutes'] - summary['doctor_minutes']
summary['utilisation_rate'] = (100.0 * summary['total_consultation_minutes'] / summary['doctor_minutes']).round(2)
summary.sort_values(by=['workload_gap'], ascending=False).tail(5)

,clinic_day,weekday,time_block,total_consultation_minutes,max_doctors,doctor_minutes,workload_gap,utilisation_rate
95,2026-06-04,Thursday,15:00-17:00,174.3,3,360.0,-185.7,48.42
86,2026-06-02,Tuesday,13:00-15:00,173.6,3,360.0,-186.4,48.22
7,2026-05-05,Tuesday,15:00-17:00,172.5,3,360.0,-187.5,47.92
30,2026-05-13,Wednesday,13:00-15:00,170.1,3,360.0,-189.9,47.25
6,2026-05-05,Tuesday,13:00-15:00,160.1,3,360.0,-199.9,44.47


##### Workload-Based Staffing Decision Rules

In [62]:
summary.sort_values(by=['utilisation_rate'], ascending=False)

,clinic_day,weekday,time_block,total_consultation_minutes,max_doctors,doctor_minutes,workload_gap,utilisation_rate
0,2026-05-04,Monday,08:00-10:00,410.3,3,360.0,50.3,113.97
99,2026-06-05,Friday,15:00-17:00,264.5,2,240.0,24.5,110.21
32,2026-05-14,Thursday,08:00-10:00,394.2,3,360.0,34.2,109.50
4,2026-05-05,Tuesday,08:00-10:00,373.8,3,360.0,13.8,103.83
85,2026-06-02,Tuesday,10:00-12:00,367.8,3,360.0,7.8,102.17
...,...,...,...,...,...,...,...,...
95,2026-06-04,Thursday,15:00-17:00,174.3,3,360.0,-185.7,48.42
86,2026-06-02,Tuesday,13:00-15:00,173.6,3,360.0,-186.4,48.22
7,2026-05-05,Tuesday,15:00-17:00,172.5,3,360.0,-187.5,47.92
30,2026-05-13,Wednesday,13:00-15:00,170.1,3,360.0,-189.9,47.25


##### Filter out utilization rate above 100% to identify workload gap issues

In [63]:
optimisation = summary[summary['utilisation_rate'] > 100].sort_values(by=['utilisation_rate'], ascending=False)

##### Optimisation the number of doctors required to ease workload gap. Allocate extra doctors with workload gap exceed 30 minutes

In [64]:
total_patients = df_all_patients.groupby(['clinic_day', 'weekday', 'time_block']).agg(doctor_minutes=('time_block_mins_spent', 'max')).reset_index()
final = optimisation.merge(total_patients, on=['clinic_day', 'weekday', 'time_block'], how='left')
final = final.rename(columns={"doctor_minutes_x": "doctor_capacity_minutes", "doctor_minutes_y": "block_minutes", "workload_gap": "workload_gap_minutes"})
final = final[final['workload_gap_minutes'] > 30]
final['extra_doctors_needed'] = (final['workload_gap_minutes'] / final['block_minutes']).round(2)
final['extra_doctors_needed'] = np.ceil(final['extra_doctors_needed'])
final

,clinic_day,weekday,time_block,total_consultation_minutes,max_doctors,doctor_capacity_minutes,workload_gap_minutes,utilisation_rate,block_minutes,extra_doctors_needed
0,2026-05-04,Monday,08:00-10:00,410.3,3,360.0,50.3,113.97,120.0,1.0
2,2026-05-14,Thursday,08:00-10:00,394.2,3,360.0,34.2,109.50,120.0,1.0


##### Compute patient penalty in minute, clinic penalty in minute, doctor cost per hour to find out the total doctor cost per block with max doctors on shift

In [65]:
dr_cost_per_hour = df_model.loc[df_model['parameter'] == 'doctor_cost_per_hour', 'value'].iloc[0]
patient_penalty_minute = df_model.loc[df_model['parameter'] == 'waiting_time_penalty_per_patient_minute', 'value'].iloc[0]
clinic_penalty_minute = df_model.loc[df_model['parameter'] == 'overtime_penalty_per_minute', 'value'].iloc[0]
df_dr['dr_cost_per_hour'] = dr_cost_per_hour
df_dr['patient_penalty_minute'] = patient_penalty_minute
df_dr['clinic_penalty_minute'] = clinic_penalty_minute
final_math = df_dr.merge(summary, on=['clinic_day', 'weekday', 'time_block'], how='left')
final_math['total_dr_cost'] = final_math['doctor_cost_per_2hr_block'] * 3
final_math['per_doctor_per_block'] = final_math['doctor_minutes'] / final_math['max_doctors']

##### Positive adjustments indicate that additional doctor capacity may be required, while negative adjustments suggest potential spare capacity that could be reduced or reallocated to other time blocks. Recommendations remain subject to minimum staffing requirements and maximum doctor availability. 

In [66]:
final_math['doctors_adjustment'] = final_math['workload_gap'] / final_math['per_doctor_per_block']
final_math['doctors_adjustment'] = np.ceil(final_math['doctors_adjustment'])
final_math_opt = final_math[['clinic_day', 'weekday', 'time_block', 'max_doctors', 'workload_gap', 'utilisation_rate', 'doctors_adjustment']].copy()
final_math_opt['theoretical_doctors_required'] = final_math_opt['max_doctors'] + final_math_opt['doctors_adjustment']
final_math_opt['feasible_recommended_doctors'] = final_math_opt['theoretical_doctors_required'].clip(lower=1, upper=final_math_opt['max_doctors'])
final_math_opt

,clinic_day,weekday,time_block,max_doctors,workload_gap,utilisation_rate,doctors_adjustment,theoretical_doctors_required,feasible_recommended_doctors
0,2026-05-04,Monday,08:00-10:00,3,50.3,113.97,1.0,4.0,3.0
1,2026-05-04,Monday,10:00-12:00,3,-34.5,90.42,-0.0,3.0,3.0
2,2026-05-04,Monday,13:00-15:00,3,-146.8,59.22,-1.0,2.0,2.0
3,2026-05-04,Monday,15:00-17:00,3,-175.4,51.28,-1.0,2.0,2.0
4,2026-05-05,Tuesday,08:00-10:00,3,13.8,103.83,1.0,4.0,3.0
...,...,...,...,...,...,...,...,...,...
95,2026-06-04,Thursday,15:00-17:00,3,-185.7,48.42,-1.0,2.0,2.0
96,2026-06-05,Friday,08:00-10:00,3,-33.0,90.83,-0.0,3.0,3.0
97,2026-06-05,Friday,10:00-12:00,3,-1.7,99.53,-0.0,3.0,3.0
98,2026-06-05,Friday,13:00-15:00,3,-144.6,59.83,-1.0,2.0,2.0


##### Set a condition for action required to management for review, should we reviewed capacity constraint, maintain current staff level or reallocate doctor capacity to other outpatient clinic

In [67]:
conditions = [
    (final_math_opt['feasible_recommended_doctors'] < final_math_opt['max_doctors']),
    (final_math_opt['theoretical_doctors_required'] == final_math_opt['max_doctors']),
    (final_math_opt['theoretical_doctors_required'] > final_math_opt['max_doctors'])
]

choices = [
    'Reallocate the doctor capacity',
    'Maintain staffing level',
    'Review capacity constraint'
]

final_math_opt['required_action'] = np.select(conditions, choices)
final_math_opt

,clinic_day,weekday,time_block,max_doctors,workload_gap,utilisation_rate,doctors_adjustment,theoretical_doctors_required,feasible_recommended_doctors,required_action
0,2026-05-04,Monday,08:00-10:00,3,50.3,113.97,1.0,4.0,3.0,Review capacity constraint
1,2026-05-04,Monday,10:00-12:00,3,-34.5,90.42,-0.0,3.0,3.0,Maintain staffing level
2,2026-05-04,Monday,13:00-15:00,3,-146.8,59.22,-1.0,2.0,2.0,Reallocate the doctor capacity
3,2026-05-04,Monday,15:00-17:00,3,-175.4,51.28,-1.0,2.0,2.0,Reallocate the doctor capacity
4,2026-05-05,Tuesday,08:00-10:00,3,13.8,103.83,1.0,4.0,3.0,Review capacity constraint
...,...,...,...,...,...,...,...,...,...,...
95,2026-06-04,Thursday,15:00-17:00,3,-185.7,48.42,-1.0,2.0,2.0,Reallocate the doctor capacity
96,2026-06-05,Friday,08:00-10:00,3,-33.0,90.83,-0.0,3.0,3.0,Maintain staffing level
97,2026-06-05,Friday,10:00-12:00,3,-1.7,99.53,-0.0,3.0,3.0,Maintain staffing level
98,2026-06-05,Friday,13:00-15:00,3,-144.6,59.83,-1.0,2.0,2.0,Reallocate the doctor capacity


##### The optimisation evaluates feasible staffing options of one doctor, two doctors and the maximum number available in each time block. For each option, the model calculates staffing cost and a penalty for consultation workload exceeding available capacity. The recommended allocation is the feasible option with the lowest total objective score.

In [68]:
dr_cost_hourly = df_model.loc[df_model['parameter'] == 'doctor_cost_per_hour', 'value'].iloc[0]
time_penalty_per_minute = df_model.loc[df_model['parameter'] == 'waiting_time_penalty_per_patient_minute', 'value'].iloc[0]

##### Assigned min doctors = 1 doctor and two doctors accordingly to compute on the clinic workload gap for 1 doctor, 2 doctors or max doctors

In [69]:
summary['per_doctor_minutes'] = summary['doctor_minutes'] / summary['max_doctors']
summary['min_doctors'] = 1
summary['two_doctors'] = 2
summary_math_optimisation = summary[['clinic_day', 'time_block', 'total_consultation_minutes', 'min_doctors', 'two_doctors', 'max_doctors', 'per_doctor_minutes']].copy()
summary_math_optimisation['min_doctors_workload_gap'] = summary_math_optimisation['total_consultation_minutes'] - (summary_math_optimisation['min_doctors'] * summary_math_optimisation['per_doctor_minutes'])
summary_math_optimisation['two_doctors_workload_gap'] = summary_math_optimisation['total_consultation_minutes'] - (summary_math_optimisation['two_doctors'] * summary_math_optimisation['per_doctor_minutes'])
summary_math_optimisation['max_doctors_workload_gap'] = summary_math_optimisation['total_consultation_minutes'] - (summary_math_optimisation['max_doctors'] * summary_math_optimisation['per_doctor_minutes'])
summary_math_optimisation['one_doctors_shortage'] = (summary_math_optimisation['min_doctors_workload_gap'].clip(lower=0))
summary_math_optimisation['two_doctors_shortage'] = (summary_math_optimisation['two_doctors_workload_gap'].clip(lower=0))
summary_math_optimisation['max_doctors_shortage'] = (summary_math_optimisation['max_doctors_workload_gap']).clip(lower=0)
summary_math_optimisation['doctor_cost'] = dr_cost_hourly
summary_math_optimisation['time_penalty'] = time_penalty_per_minute

#### For one doctor

In [70]:
summary_math_optimisation['min_doctor_per_block'] = summary_math_optimisation['min_doctors'] * summary_math_optimisation['doctor_cost'] * 2
summary_math_optimisation['min_doctor_penalty_per_block'] = summary_math_optimisation['one_doctors_shortage'] * summary_math_optimisation['time_penalty']
summary_math_optimisation['min_overall_cost'] = summary_math_optimisation['min_doctor_per_block'] + summary_math_optimisation['min_doctor_penalty_per_block']

summary_math_optimisation

,clinic_day,time_block,total_consultation_minutes,min_doctors,two_doctors,max_doctors,per_doctor_minutes,min_doctors_workload_gap,two_doctors_workload_gap,max_doctors_workload_gap,one_doctors_shortage,two_doctors_shortage,max_doctors_shortage,doctor_cost,time_penalty,min_doctor_per_block,min_doctor_penalty_per_block,min_overall_cost
0,2026-05-04,08:00-10:00,410.3,1,2,3,120.0,290.3,170.3,50.3,290.3,170.3,50.3,120,2,240,580.6,820.6
1,2026-05-04,10:00-12:00,325.5,1,2,3,120.0,205.5,85.5,-34.5,205.5,85.5,0.0,120,2,240,411.0,651.0
2,2026-05-04,13:00-15:00,213.2,1,2,3,120.0,93.2,-26.8,-146.8,93.2,0.0,0.0,120,2,240,186.4,426.4
3,2026-05-04,15:00-17:00,184.6,1,2,3,120.0,64.6,-55.4,-175.4,64.6,0.0,0.0,120,2,240,129.2,369.2
4,2026-05-05,08:00-10:00,373.8,1,2,3,120.0,253.8,133.8,13.8,253.8,133.8,13.8,120,2,240,507.6,747.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026-06-04,15:00-17:00,174.3,1,2,3,120.0,54.3,-65.7,-185.7,54.3,0.0,0.0,120,2,240,108.6,348.6
96,2026-06-05,08:00-10:00,327.0,1,2,3,120.0,207.0,87.0,-33.0,207.0,87.0,0.0,120,2,240,414.0,654.0
97,2026-06-05,10:00-12:00,358.3,1,2,3,120.0,238.3,118.3,-1.7,238.3,118.3,0.0,120,2,240,476.6,716.6
98,2026-06-05,13:00-15:00,215.4,1,2,3,120.0,95.4,-24.6,-144.6,95.4,0.0,0.0,120,2,240,190.8,430.8


##### For two doctors

In [71]:
summary_math_optimisation['two_doctors_per_block'] = summary_math_optimisation['two_doctors'] * summary_math_optimisation['doctor_cost'] * 2
summary_math_optimisation['two_doctors_penalty_per_block'] = summary_math_optimisation['two_doctors_shortage'] * summary_math_optimisation['time_penalty']
summary_math_optimisation['two_overall_cost'] = summary_math_optimisation['two_doctors_per_block'] + summary_math_optimisation['two_doctors_penalty_per_block']

##### For max doctors

In [72]:
summary_math_optimisation['max_doctors_per_block'] = summary_math_optimisation['max_doctors'] * summary_math_optimisation['doctor_cost'] * 2
summary_math_optimisation['max_doctors_penalty_per_block'] = summary_math_optimisation['max_doctors_shortage'] * summary_math_optimisation['time_penalty']
summary_math_optimisation['max_overall_cost'] = summary_math_optimisation['max_doctors_per_block'] + summary_math_optimisation['max_doctors_penalty_per_block']
summary_math_optimisation

,clinic_day,time_block,total_consultation_minutes,min_doctors,two_doctors,max_doctors,per_doctor_minutes,min_doctors_workload_gap,two_doctors_workload_gap,max_doctors_workload_gap,...,time_penalty,min_doctor_per_block,min_doctor_penalty_per_block,min_overall_cost,two_doctors_per_block,two_doctors_penalty_per_block,two_overall_cost,max_doctors_per_block,max_doctors_penalty_per_block,max_overall_cost
0,2026-05-04,08:00-10:00,410.3,1,2,3,120.0,290.3,170.3,50.3,...,2,240,580.6,820.6,480,340.6,820.6,720,100.6,820.6
1,2026-05-04,10:00-12:00,325.5,1,2,3,120.0,205.5,85.5,-34.5,...,2,240,411.0,651.0,480,171.0,651.0,720,0.0,720.0
2,2026-05-04,13:00-15:00,213.2,1,2,3,120.0,93.2,-26.8,-146.8,...,2,240,186.4,426.4,480,0.0,480.0,720,0.0,720.0
3,2026-05-04,15:00-17:00,184.6,1,2,3,120.0,64.6,-55.4,-175.4,...,2,240,129.2,369.2,480,0.0,480.0,720,0.0,720.0
4,2026-05-05,08:00-10:00,373.8,1,2,3,120.0,253.8,133.8,13.8,...,2,240,507.6,747.6,480,267.6,747.6,720,27.6,747.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026-06-04,15:00-17:00,174.3,1,2,3,120.0,54.3,-65.7,-185.7,...,2,240,108.6,348.6,480,0.0,480.0,720,0.0,720.0
96,2026-06-05,08:00-10:00,327.0,1,2,3,120.0,207.0,87.0,-33.0,...,2,240,414.0,654.0,480,174.0,654.0,720,0.0,720.0
97,2026-06-05,10:00-12:00,358.3,1,2,3,120.0,238.3,118.3,-1.7,...,2,240,476.6,716.6,480,236.6,716.6,720,0.0,720.0
98,2026-06-05,13:00-15:00,215.4,1,2,3,120.0,95.4,-24.6,-144.6,...,2,240,190.8,430.8,480,0.0,480.0,720,0.0,720.0


##### Recommendation for mathematical optimization - When preferably one-doctor and two-doctor options have the same objective score, recommend two doctors to reduce workload pressure without automatically selecting maximum staffing.

In [73]:
objective = ['min_overall_cost', 'two_overall_cost', 'max_overall_cost']
summary_math_optimisation['lowest_score'] = (summary_math_optimisation[objective].min(axis=1))

one_is_lowest = np.isclose(
    summary_math_optimisation['min_overall_cost'],
    summary_math_optimisation['lowest_score'])

two_is_lowest = np.isclose(
    summary_math_optimisation['two_overall_cost'],
    summary_math_optimisation['lowest_score'])

max_is_lowest = np.isclose(
    summary_math_optimisation['max_overall_cost'],
    summary_math_optimisation['lowest_score'])

conditions = [
    two_is_lowest,
    one_is_lowest,
    max_is_lowest
]

choices = [
    summary_math_optimisation['two_doctors'],
    summary_math_optimisation['min_doctors'],
    summary_math_optimisation['max_doctors']
]

option_choices = [
    'two_doctors',
    'min_doctors',
    'mmax_doctors'
]

summary_math_optimisation['recommended_doctors'] = np.select(conditions, choices)
summary_math_optimisation['best_option'] = np.select(conditions, option_choices)

summary_math_optimisation

,clinic_day,time_block,total_consultation_minutes,min_doctors,two_doctors,max_doctors,per_doctor_minutes,min_doctors_workload_gap,two_doctors_workload_gap,max_doctors_workload_gap,...,min_overall_cost,two_doctors_per_block,two_doctors_penalty_per_block,two_overall_cost,max_doctors_per_block,max_doctors_penalty_per_block,max_overall_cost,lowest_score,recommended_doctors,best_option
0,2026-05-04,08:00-10:00,410.3,1,2,3,120.0,290.3,170.3,50.3,...,820.6,480,340.6,820.6,720,100.6,820.6,820.6,2,two_doctors
1,2026-05-04,10:00-12:00,325.5,1,2,3,120.0,205.5,85.5,-34.5,...,651.0,480,171.0,651.0,720,0.0,720.0,651.0,2,two_doctors
2,2026-05-04,13:00-15:00,213.2,1,2,3,120.0,93.2,-26.8,-146.8,...,426.4,480,0.0,480.0,720,0.0,720.0,426.4,1,min_doctors
3,2026-05-04,15:00-17:00,184.6,1,2,3,120.0,64.6,-55.4,-175.4,...,369.2,480,0.0,480.0,720,0.0,720.0,369.2,1,min_doctors
4,2026-05-05,08:00-10:00,373.8,1,2,3,120.0,253.8,133.8,13.8,...,747.6,480,267.6,747.6,720,27.6,747.6,747.6,2,two_doctors
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026-06-04,15:00-17:00,174.3,1,2,3,120.0,54.3,-65.7,-185.7,...,348.6,480,0.0,480.0,720,0.0,720.0,348.6,1,min_doctors
96,2026-06-05,08:00-10:00,327.0,1,2,3,120.0,207.0,87.0,-33.0,...,654.0,480,174.0,654.0,720,0.0,720.0,654.0,2,two_doctors
97,2026-06-05,10:00-12:00,358.3,1,2,3,120.0,238.3,118.3,-1.7,...,716.6,480,236.6,716.6,720,0.0,720.0,716.6,2,two_doctors
98,2026-06-05,13:00-15:00,215.4,1,2,3,120.0,95.4,-24.6,-144.6,...,430.8,480,0.0,480.0,720,0.0,720.0,430.8,1,min_doctors


#### Prepare Patient Arrival and Conusltation Duration Inputs

In [74]:
patients_arrived = pd.to_datetime(df_all_patients['actual_arrival_time'], format='%H:%M')
appt_time = pd.to_datetime(df_all_patients['appointment_time'], format='%H:%M')
con_minutes = pd.to_timedelta(df_all_patients['consultation_minutes'], unit='m')

df_all_patients['arrival_delay_minutes'] = (patients_arrived - appt_time).dt.total_seconds() / 60
df_all_patients['scheduled_no_wait_end_time'] = appt_time + con_minutes
df_all_patients['scheduled_no_wait_end_time'] = df_all_patients['scheduled_no_wait_end_time'].dt.strftime('%H:%M')
df_all_patients

,patient_id,clinic_day,weekday,clinic_site,appointment_time,appointment_minute,time_block,no_show,actual_arrival_time,actual_arrival_minute,...,age_band,visit_type,service_type,complexity,skill_required,registration_minutes,consultation_minutes,hours_spent,time_block_mins_spent,scheduled_no_wait_end_time
0,P00001,2026-05-04,Monday,Clinic A,08:00,480,08:00-10:00,0,07:45,465.0,...,Adult,New,General Review,Low,General,6.0,13.4,0 days 02:00:00,120.0,08:13
1,P00002,2026-05-04,Monday,Clinic A,08:08,488,08:00-10:00,0,07:59,479.0,...,Adult,Follow-up,Complex Care Review,Medium,Complex,5.1,48.0,0 days 02:00:00,120.0,08:56
2,P00003,2026-05-04,Monday,Clinic A,08:16,496,08:00-10:00,0,08:15,495.0,...,Adult,Follow-up,Physiotherapy Review,Low,Rehab,7.8,30.8,0 days 02:00:00,120.0,08:46
3,P00004,2026-05-04,Monday,Clinic A,08:24,504,08:00-10:00,0,08:17,497.0,...,Adult,Follow-up,Complex Care Review,Low,Complex,5.8,40.3,0 days 02:00:00,120.0,09:04
4,P00005,2026-05-04,Monday,Clinic B,08:32,512,08:00-10:00,0,08:36,516.0,...,Adult,Follow-up,Post-op Follow-up,Low,General,5.3,16.0,0 days 02:00:00,120.0,08:48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1345,P01346,2026-06-05,Friday,Clinic A,16:10,970,15:00-17:00,1,NaN,NaN,...,Adult,New,Post-op Follow-up,Medium,General,NaN,NaN,0 days 02:00:00,120.0,NaN
1346,P01347,2026-06-05,Friday,Clinic A,16:20,980,15:00-17:00,0,16:23,983.0,...,Adult,Follow-up,Post-op Follow-up,Medium,General,5.5,29.4,0 days 02:00:00,120.0,16:49
1347,P01348,2026-06-05,Friday,Clinic A,16:30,990,15:00-17:00,0,16:31,991.0,...,Senior,New,Physiotherapy Review,Medium,Rehab,5.5,31.4,0 days 02:00:00,120.0,17:01
1348,P01349,2026-06-05,Friday,Clinic B,16:40,1000,15:00-17:00,0,16:44,1004.0,...,Adult,New,Post-op Follow-up,Medium,General,5.2,17.8,0 days 02:00:00,120.0,16:57


#### Arrival delay is calculated as the difference between actual arrival time and scheduled appointment time. It measures whether patients arrive early or late and should not be interpreted as patient waiting time.

In [75]:
planned_dt = pd.to_datetime(df_all_patients['scheduled_no_wait_end_time'], format='%H:%M', errors='coerce')
actual_dt = pd.to_datetime(df_all_patients['actual_arrival_time'], format='%H:%M', errors='coerce')
df_all_patients['time_diff'] = (planned_dt - actual_dt).dt.total_seconds() / 60
df_all_patients.sort_values(by='time_diff', ascending=True)

,patient_id,clinic_day,weekday,clinic_site,appointment_time,appointment_minute,time_block,no_show,actual_arrival_time,actual_arrival_minute,...,visit_type,service_type,complexity,skill_required,registration_minutes,consultation_minutes,hours_spent,time_block_mins_spent,scheduled_no_wait_end_time,time_diff
343,P00344,2026-05-12,Tuesday,Clinic A,10:32,632,10:00-12:00,0,10:51,651.0,...,New,General Review,Medium,General,4.4,11.8,0 days 02:00:00,120.0,10:43,-8.0
672,P00673,2026-05-20,Wednesday,Clinic A,11:12,672,10:00-12:00,0,11:32,692.0,...,Follow-up,General Review,Medium,General,8.3,12.3,0 days 02:00:00,120.0,11:24,-8.0
311,P00312,2026-05-11,Monday,Clinic A,14:50,890,13:00-15:00,0,15:12,912.0,...,Follow-up,General Review,Medium,General,6.5,15.7,0 days 02:00:00,120.0,15:05,-7.0
203,P00204,2026-05-07,Thursday,Clinic B,14:50,890,13:00-15:00,0,15:10,910.0,...,New,General Review,Low,General,4.8,14.7,0 days 02:00:00,120.0,15:04,-6.0
530,P00531,2026-05-15,Friday,Clinic B,15:20,920,15:00-17:00,0,15:39,939.0,...,Follow-up,General Review,Low,General,4.2,13.5,0 days 02:00:00,120.0,15:33,-6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1333,P01334,2026-06-05,Friday,Clinic B,14:10,850,13:00-15:00,1,NaN,NaN,...,New,Post-op Follow-up,Low,General,NaN,NaN,0 days 02:00:00,120.0,NaN,NaN
1335,P01336,2026-06-05,Friday,Clinic A,14:30,870,13:00-15:00,1,NaN,NaN,...,New,General Review,Low,General,NaN,NaN,0 days 02:00:00,120.0,NaN,NaN
1337,P01338,2026-06-05,Friday,Clinic B,14:50,890,13:00-15:00,1,NaN,NaN,...,Follow-up,Physiotherapy Review,Low,Rehab,NaN,NaN,0 days 02:00:00,120.0,NaN,NaN
1342,P01343,2026-06-05,Friday,Clinic A,15:40,940,15:00-17:00,1,NaN,NaN,...,Follow-up,Post-op Follow-up,Medium,General,NaN,NaN,0 days 02:00:00,120.0,NaN,NaN


##### Filter out patients with no show

In [76]:
df_patients = df_all_patients.copy()
df_patients = df_patients[df_patients['no_show'] == 0]
df_patients

,patient_id,clinic_day,weekday,clinic_site,appointment_time,appointment_minute,time_block,no_show,actual_arrival_time,actual_arrival_minute,...,visit_type,service_type,complexity,skill_required,registration_minutes,consultation_minutes,hours_spent,time_block_mins_spent,scheduled_no_wait_end_time,time_diff
0,P00001,2026-05-04,Monday,Clinic A,08:00,480,08:00-10:00,0,07:45,465.0,...,New,General Review,Low,General,6.0,13.4,0 days 02:00:00,120.0,08:13,28.0
1,P00002,2026-05-04,Monday,Clinic A,08:08,488,08:00-10:00,0,07:59,479.0,...,Follow-up,Complex Care Review,Medium,Complex,5.1,48.0,0 days 02:00:00,120.0,08:56,57.0
2,P00003,2026-05-04,Monday,Clinic A,08:16,496,08:00-10:00,0,08:15,495.0,...,Follow-up,Physiotherapy Review,Low,Rehab,7.8,30.8,0 days 02:00:00,120.0,08:46,31.0
3,P00004,2026-05-04,Monday,Clinic A,08:24,504,08:00-10:00,0,08:17,497.0,...,Follow-up,Complex Care Review,Low,Complex,5.8,40.3,0 days 02:00:00,120.0,09:04,47.0
4,P00005,2026-05-04,Monday,Clinic B,08:32,512,08:00-10:00,0,08:36,516.0,...,Follow-up,Post-op Follow-up,Low,General,5.3,16.0,0 days 02:00:00,120.0,08:48,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1344,P01345,2026-06-05,Friday,Clinic A,16:00,960,15:00-17:00,0,15:54,954.0,...,Follow-up,Complex Care Review,Medium,Complex,5.9,32.7,0 days 02:00:00,120.0,16:32,38.0
1346,P01347,2026-06-05,Friday,Clinic A,16:20,980,15:00-17:00,0,16:23,983.0,...,Follow-up,Post-op Follow-up,Medium,General,5.5,29.4,0 days 02:00:00,120.0,16:49,26.0
1347,P01348,2026-06-05,Friday,Clinic A,16:30,990,15:00-17:00,0,16:31,991.0,...,New,Physiotherapy Review,Medium,Rehab,5.5,31.4,0 days 02:00:00,120.0,17:01,30.0
1348,P01349,2026-06-05,Friday,Clinic B,16:40,1000,15:00-17:00,0,16:44,1004.0,...,New,Post-op Follow-up,Medium,General,5.2,17.8,0 days 02:00:00,120.0,16:57,13.0


##### Patients in different scenarios with number of doctors assigned

In [77]:
time_block_columns=[
    "08:00-10:00",
    "10:00-12:00",
    "13:00-15:00",
    "15:00-17:00"
]

df_staff_long = df_staff.melt(id_vars=['scenario', 'description'], value_vars=time_block_columns, var_name='time_block', value_name='doctors_assigned')
df_patients_scenario = df_patient.merge(df_staff_long, on='time_block', how='left')

#### Data quality check on scenarios

In [78]:
df_patients_scenario.isna().sum()

patient_id               0
clinic_day               0
weekday                  0
clinic_site              0
appointment_time         0
appointment_minute       0
time_block               0
no_show                  0
actual_arrival_time      0
actual_arrival_minute    0
arrival_delay_minutes    0
priority                 0
age_band                 0
visit_type               0
service_type             0
complexity               0
skill_required           0
registration_minutes     0
consultation_minutes     0
scenario                 0
description              0
doctors_assigned         0
dtype: int64

##### Two staffing scenarios are evaluated using the same patient demand, clinic day and time block. The only factor changed is the number of doctors assigned, allowing differences in waiting time to be attributed to staffing capacity.

- Scenario 1: Baseline of 2 doctors all day on 2026-05-04 between 08:00 - 10:00 
- Scenario 2: Additional-capacity staffing with three doctors for the same patients and period

In [79]:
group = df_patients_scenario[
    (df_patients_scenario['scenario'] == 'Baseline_2_Doctors_All_Day') &
    (df_patients_scenario['clinic_day'] == '2026-05-04') &
    (df_patients_scenario['time_block'] == '08:00-10:00')
].sort_values('actual_arrival_minute').copy()

In [80]:
group_next = df_patients_scenario[
    (df_patients_scenario['scenario'] == 'Extra_3_Doctors_All_Day') &
    (df_patients_scenario['clinic_day'] == '2026-05-04') &
    (df_patients_scenario['time_block'] == '08:00-10:00')
].copy()

##### Number of doctors

In [81]:
no_of_dr = int(group['doctors_assigned'].iloc[0])
no_of_dr_next = int(group_next['doctors_assigned'].iloc[0])

#### Create number of doctor availability times

In [82]:
dr_availability = [0.0] * no_of_dr
dr_availability_next = [0.0] * no_of_dr_next

##### Create empty lists for results

In [83]:
start_times = []
waiting_times = []
end_times = []

start_times_next = []
waiting_times_next = []
end_times_next = []

##### Patients are processed in order of actual arrival time. Each patient is assigned to the doctor who becomes available first. If a doctor is free when the patient arrives, consultation begins immediately. Otherwise, the patient waits until the earliest doctor becomes available.

#### Run Simulation on first scenario

In [84]:
for _, patient in group.iterrows():
    
    arrival = float(patient['actual_arrival_minute'])
    duration = float(patient['consultation_minutes'])
    doctor = int(np.argmin(dr_availability))

    start = max(arrival, dr_availability[doctor])
    wait = start - arrival
    end = start + duration
    
    dr_availability[doctor] = end

    start_times.append(start)
    waiting_times.append(wait)
    end_times.append(end)


In [85]:
group['simulated_start_minute'] = start_times
group['simulated_waiting_minute'] = waiting_times
group['simulated_end_minute'] = end_times

In [86]:
group['no_wait_end_minute'] = group['actual_arrival_minute'] + group['consultation_minutes']
group[['patient_id', 'actual_arrival_minute', 'consultation_minutes', 'no_wait_end_minute', 'simulated_start_minute', 'simulated_waiting_minute', 'simulated_end_minute']]
group['average_time'] = group['simulated_waiting_minute'].mean().round(2)
group['median_time'] = group['simulated_waiting_minute'].median()
group['max_wait_time'] = group['simulated_waiting_minute'].max()
group['target_p90_wait_time'] = 45
group

,patient_id,clinic_day,weekday,clinic_site,appointment_time,appointment_minute,time_block,no_show,actual_arrival_time,actual_arrival_minute,...,description,doctors_assigned,simulated_start_minute,simulated_waiting_minute,simulated_end_minute,no_wait_end_minute,average_time,median_time,max_wait_time,target_p90_wait_time
0,P00001,2026-05-04,Monday,Clinic A,08:00,480,08:00-10:00,0,07:45,465.0,...,Current-state baseline with two doctors throug...,2,465.0,0.0,478.4,478.4,37.43,47.2,62.9,45
5,P00002,2026-05-04,Monday,Clinic A,08:08,488,08:00-10:00,0,07:59,479.0,...,Current-state baseline with two doctors throug...,2,479.0,0.0,527.0,527.0,37.43,47.2,62.9,45
10,P00003,2026-05-04,Monday,Clinic A,08:16,496,08:00-10:00,0,08:15,495.0,...,Current-state baseline with two doctors throug...,2,495.0,0.0,525.8,525.8,37.43,47.2,62.9,45
15,P00004,2026-05-04,Monday,Clinic A,08:24,504,08:00-10:00,0,08:17,497.0,...,Current-state baseline with two doctors throug...,2,525.8,28.8,566.1,537.3,37.43,47.2,62.9,45
25,P00006,2026-05-04,Monday,Clinic A,08:40,520,08:00-10:00,0,08:35,515.0,...,Current-state baseline with two doctors throug...,2,527.0,12.0,564.8,552.8,37.43,47.2,62.9,45
20,P00005,2026-05-04,Monday,Clinic B,08:32,512,08:00-10:00,0,08:36,516.0,...,Current-state baseline with two doctors throug...,2,564.8,48.8,580.8,532.0,37.43,47.2,62.9,45
35,P00008,2026-05-04,Monday,Clinic A,08:56,536,08:00-10:00,0,08:37,517.0,...,Current-state baseline with two doctors throug...,2,566.1,49.1,585.6,536.5,37.43,47.2,62.9,45
30,P00007,2026-05-04,Monday,Clinic B,08:48,528,08:00-10:00,0,08:38,518.0,...,Current-state baseline with two doctors throug...,2,580.8,62.8,607.5,544.7,37.43,47.2,62.9,45
45,P00010,2026-05-04,Monday,Clinic A,09:12,552,08:00-10:00,0,09:01,541.0,...,Current-state baseline with two doctors throug...,2,585.6,44.6,601.9,557.3,37.43,47.2,62.9,45
50,P00012,2026-05-04,Monday,Clinic A,09:28,568,08:00-10:00,0,09:16,556.0,...,Current-state baseline with two doctors throug...,2,601.9,45.9,635.6,589.7,37.43,47.2,62.9,45


##### Run simulation on second scenario

In [87]:
for _, patient_next in group_next.iterrows():

    arrival = float(patient_next['actual_arrival_minute'])
    duration = float(patient_next['consultation_minutes'])
    doctor = int(np.argmin(dr_availability_next))

    start = max(arrival, dr_availability_next[doctor])
    wait = start - arrival
    end = start + duration

    dr_availability_next[doctor] = end

    start_times_next.append(start)
    waiting_times_next.append(wait)
    end_times_next.append(end)

In [88]:
group_next['simulated_start_minute'] = start_times_next
group_next['simulated_waiting_minute'] = waiting_times_next
group_next['simulated_end_minute'] = end_times_next

In [89]:
group_next['no_wait_end_minute'] = group_next['actual_arrival_minute'] + group_next['consultation_minutes']   
group_next[['patient_id', 'actual_arrival_minute', 'consultation_minutes', 'no_wait_end_minute', 'simulated_start_minute', 'simulated_waiting_minute', 'simulated_end_minute']]

,patient_id,actual_arrival_minute,consultation_minutes,no_wait_end_minute,simulated_start_minute,simulated_waiting_minute,simulated_end_minute
1,P00001,465.0,13.4,478.4,465.0,0.0,478.4
6,P00002,479.0,48.0,527.0,479.0,0.0,527.0
11,P00003,495.0,30.8,525.8,495.0,0.0,525.8
16,P00004,497.0,40.3,537.3,497.0,0.0,537.3
21,P00005,516.0,16.0,532.0,525.8,9.8,541.8
26,P00006,515.0,37.8,552.8,527.0,12.0,564.8
31,P00007,518.0,26.7,544.7,537.3,19.3,564.0
36,P00008,517.0,19.5,536.5,541.8,24.8,561.3
41,P00009,559.0,23.5,582.5,561.3,2.3,584.8
46,P00010,541.0,16.3,557.3,564.0,23.0,580.3


In [90]:
group_next['average_time'] = group_next['simulated_waiting_minute'].mean()
group_next['median_time'] = group_next['simulated_waiting_minute'].median()
group_next['max_wait_time'] = group_next['simulated_waiting_minute'].max()
group_next['target_p90_wait_time'] = 45
group_next



,patient_id,clinic_day,weekday,clinic_site,appointment_time,appointment_minute,time_block,no_show,actual_arrival_time,actual_arrival_minute,...,description,doctors_assigned,simulated_start_minute,simulated_waiting_minute,simulated_end_minute,no_wait_end_minute,average_time,median_time,max_wait_time,target_p90_wait_time
1,P00001,2026-05-04,Monday,Clinic A,08:00,480,08:00-10:00,0,07:45,465.0,...,Adds one doctor across all clinic blocks; stro...,3,465.0,0.0,478.4,478.4,8.507143,8.55,24.8,45
6,P00002,2026-05-04,Monday,Clinic A,08:08,488,08:00-10:00,0,07:59,479.0,...,Adds one doctor across all clinic blocks; stro...,3,479.0,0.0,527.0,527.0,8.507143,8.55,24.8,45
11,P00003,2026-05-04,Monday,Clinic A,08:16,496,08:00-10:00,0,08:15,495.0,...,Adds one doctor across all clinic blocks; stro...,3,495.0,0.0,525.8,525.8,8.507143,8.55,24.8,45
16,P00004,2026-05-04,Monday,Clinic A,08:24,504,08:00-10:00,0,08:17,497.0,...,Adds one doctor across all clinic blocks; stro...,3,497.0,0.0,537.3,537.3,8.507143,8.55,24.8,45
21,P00005,2026-05-04,Monday,Clinic B,08:32,512,08:00-10:00,0,08:36,516.0,...,Adds one doctor across all clinic blocks; stro...,3,525.8,9.8,541.8,532.0,8.507143,8.55,24.8,45
26,P00006,2026-05-04,Monday,Clinic A,08:40,520,08:00-10:00,0,08:35,515.0,...,Adds one doctor across all clinic blocks; stro...,3,527.0,12.0,564.8,552.8,8.507143,8.55,24.8,45
31,P00007,2026-05-04,Monday,Clinic B,08:48,528,08:00-10:00,0,08:38,518.0,...,Adds one doctor across all clinic blocks; stro...,3,537.3,19.3,564.0,544.7,8.507143,8.55,24.8,45
36,P00008,2026-05-04,Monday,Clinic A,08:56,536,08:00-10:00,0,08:37,517.0,...,Adds one doctor across all clinic blocks; stro...,3,541.8,24.8,561.3,536.5,8.507143,8.55,24.8,45
41,P00009,2026-05-04,Monday,Clinic B,09:04,544,08:00-10:00,0,09:19,559.0,...,Adds one doctor across all clinic blocks; stro...,3,561.3,2.3,584.8,582.5,8.507143,8.55,24.8,45
46,P00010,2026-05-04,Monday,Clinic A,09:12,552,08:00-10:00,0,09:01,541.0,...,Adds one doctor across all clinic blocks; stro...,3,564.0,23.0,580.3,557.3,8.507143,8.55,24.8,45


##### The baseline two-doctor scenario produces longer simulated waiting times as consultations accumulate and both doctors remain occupied. The three-doctors scenario is expected to reduce average and upper-percentile waiting times by increasing available consultation capacity. The final comparison evaluates whether this service improvement justifies the additional staffing cost.